# Bit-Banging SWD with `scope.bitbanger`
`scope.bitbanger` can be made to speak many different protocols. In this example, we'll have it speak Arm's serial-wire-debug (SWD).

Works out-of-the-box with SAM4S and STM32F3 targets. Other Arm targets may require some modifications -- consult their documentation.

In [ ]:
SCOPE = 'OPENADC'
PLATFORM = 'CW308_SAM4S'
#PLATFORM = 'CW308_STM32F3'

In [ ]:
%run '/home/jpnewae/git/cw_trimmed/jupyter/Setup_Scripts/Setup_Generic.ipynb'

We'll use our usual `simpleserial-trace` firmware because it allows us to read or write debug registers via the simpleserial interface, which we can use to verify that writing and reading debug registers over SWD with `scope.bitbanger` works as expected.

In [ ]:
cw.program_target(scope, prog, "../../../firmware/mcu/simpleserial-trace/simpleserial-trace-{}.hex".format(PLATFORM))

## Connect TraceWhisperer

Let's connect the target to TraceWhisperer so that we can query its debug registers.

If you're not familiar with TraceWhisperer, you just need to know that here we are not using its main features; we are using only one small part of TraceWhisperer: its ability to get/set the target's debug register over UART (this is not black magic, it works because the firmware has SimpleSerial commands that support this).

We don't need TraceWhisperer to use `scope.bitbanger` on an SWD interface; we are using it here because it makes our demo easier.

In [ ]:
import time
def target_reset():
    scope.io.nrst = 'low'
    time.sleep(0.05)
    scope.io.nrst = 'high'
    time.sleep(0.05)

In [ ]:
target_reset()

In [ ]:
target_reset()
target.flush()
target.write('x\n')
time.sleep(0.2)
resp = target.read()
target.simpleserial_write('i', b'')
time.sleep(0.1)
response = target.read().split('\n')[0]
print(response)
#assert response == 'ChipWhisperer simpleserial-trace, compiled Sep  2 2022, 13:55:43'

Since we'll be modifying these via SWD, we need to disable the caching that TraceWhisperer does by default.

In [ ]:
scope.trace.target = target
scope.trace.target_registers.use_register_cache = False
scope.trace.target_registers

## Set Up BitBanger

`set_defaults()` sets some useful default SWD settings for us:

In [ ]:
scope.bitbanger.swd.set_defaults()
print(scope.bitbanger)

We still need to specify a few things:
- the data and clock pins
- the clock rate

In [ ]:
scope.bitbanger.data_pin = 'USERIO_D0'
scope.bitbanger.clock_pin = 'USERIO_D1'
scope.bitbanger.continuous_clk = True
scope.bitbanger.clk_div = 80

`scope.bitbanger` uses the ADC clock (not the the target clock!) and divides it by `scope.bitbanger.clk_div` to drive bits out.

SWD interfaces support a wide range of clock frequencies; it's up to you what to set this to.

Setting it to 80 gives us this rate:

In [ ]:
print('Rate: %0.1f bps' % (scope.clock.adc_freq / scope.bitbanger.clk_div))

We specified the data and clock pins as `USERIO_D0` and `USERIO_D1`; **you need to manually jumper those to the TMS and TCK header pins on your CW313 or CW308 target board.**

Arm processors' debug port usually boots up in JTAG mode; a JTAG-to-SWD sequence must be sent to switch over to SWD.

`wake_swd()` does this for us; it also reads and returns the device's reported ID code.

*Note 1: if this fails: did you connect TMS and TCK as directed above?*

*Note 2: if you are using a different target, additional setup may be required to enable SWD. Consult your target documentation. For example, our STM32 target requires its SWO pin to be enabled (see [code](https://github.com/newaetech/chipwhisperer/blob/develop/firmware/mcu/simpleserial-trace/simpleserial-trace.c#L109)).*

In [ ]:
target_reset()
idcode = scope.bitbanger.swd.wake_swd()
print(hex(idcode))

If you know your device's ID code, you can have `wake_swd()` check it for you.

Let's make that fail on purpose:

In [ ]:
scope.bitbanger.swd.wake_swd(expected_id_code=idcode ^ 0x100) # should fail because we're expecting the wrong ID code!

With the correct ID code, there is no output. You can use this to validate that communication on the link is successful:

In [ ]:
scope.bitbanger.swd.wake_swd(expected_id_code=idcode)

Now let's use `scope.bitbanger.read` to read target memory.

First, you need to figure out what address you want to read.

In our case, we're interested in the debug registers. Let's pick the `DWT_COMP0` since it's something we can safely write without breaking anything. To find its address, you'll need to dive into HAL header files.

For the SAM4S, [this file](https://github.com/newaetech/chipwhisperer/blob/develop/firmware/mcu/hal/sam4s/inc/core_cm4.h) tells you that it's at offset `0x020` of the DWT, which itself is at `0xE0001000` (p.s.: it's the same address for the STM32F3).

In [ ]:
rdata = scope.bitbanger.swd.read(0xE0001020)
rstring = '%08x' % rdata
assert rstring == scope.trace.target_registers.DWT_COMP0

If you want to check that the address contents is some expected value, you can do this:

In [ ]:
scope.bitbanger.swd.read(address=0xE0001020, expected_data=0x1d60)

You can also write to a target memory address:

In [ ]:
scope.bitbanger.swd.write(0xE0001020, 0x12345678)

Let's verify that two ways:

In [ ]:
assert scope.trace.target_registers.DWT_COMP0 == '12345678'

In [ ]:
scope.bitbanger.swd.read(0xE0001020, 0x12345678)

# Bitbanger-triggered traces

If this was all you could do with `scope.bitbanger.swd`, all you would have is a rather expensive, very low-level debug tool.

Which could be cool if you want to *really* understand how SWD works at the bit-level!

But that's not why we created `scope.bitbanger`; we made it so that you could do this:

In [ ]:
scope.trigger.module = 'bitbanger'

Remember how `scope.bitbanger` outputs are driven by a divided-down version of the ADC sampling clock? This means we can obtain power traces that are **synchronous with the SWD clock**, and synchronized to a precise event in our bit-banged SWD conversation.

And since the bit-banging's timing is fully hardware-driven (with no software control involved), its timing is fully and perfectly repeatable.

Our `scope.bitbanger.swd`methods have a triggering option that we'll now use to specify when we want to generate a trigger.

Let's define a handy function which lets us capture a trace on arbitrary commands:

In [ ]:
def capture_trace(sendcommand):
    scope.arm()
    sendcommand()
    ret = scope.capture()
    if ret:
        print("WARNING: Timeout happened during capture")
        return None
    wave = scope.get_last_trace()
    return wave

In [ ]:
scope.adc.samples = 100000
scope.adc.presamples = 0

Now we capture a trace synchronized to a memory write:

In [ ]:
scope.gain.db = 23

In [ ]:
trace = capture_trace(lambda: scope.bitbanger.swd.write(0xE0001020, 0x12345678, trigger_bits=[0]))

Let's average several captures to get a cleaner trace:

In [ ]:
import numpy as np
from tqdm.notebook import tnrange

avgtrace = np.zeros(scope.adc.samples)
for i in tnrange(500):
    trace = capture_trace(lambda: scope.bitbanger.swd.write(0xE0001020, 0xFFFFFFFF, trigger_bits=[0]))
    avgtrace += trace

In [ ]:
cw.plot(avgtrace)

What are we looking at here? Since we triggered right at the start of the bit-banged data, it would be helpful to know how long it took to send that data.

You can break out a logic analyzer and look at the bits on the wire, or you peak inside `scope.bitbanger.swd.write()`; let's do the latter.

A `BitBangerPacket` object is initialized, then extending by 5 calls to `scope.bitbanger.swd.getpacket()` to grow (extend) this `BitBangerPacket`; then it is dispatched via `sendpacket()`:

In [ ]:
scope.bitbanger.swd.write??

Now, `sendpacket()` stores what it sends into `scope.bitbanger.last_packet_sent`; we can look at that to learn that this bit-banged packet was 253 bits long:

In [ ]:
scope.bitbanger.last_packet_sent.num_bits

And since each bit-banged "bit" is `scope.bitbanger.clk_div` ADC samples, we calculate the duration of our bit-banged packet in ADC clock samples like so:

In [ ]:
scope.bitbanger.last_packet_sent.num_bits * scope.bitbanger.clk_div

This may seem a bit involved, but that's what you should expect with `scope.bitbanger`: you are working at the bit level, so you need to understand what's going on at the bit level. `scope.bitbanger.swd` provides example/convenience functions to help you get started and show what's possible. You are expected to expand on these examples to meet your needs.

Back to our plot: at first glance you may not find very much that seems interesting, but there is a larger negative spike around sample 20240, which is shortly after our write command is complete.

*(These observations are for the SAM4S target. If you're using the STM32F3, the spikes are much more prominent.)*

These power traces may not seem very compelling, but realize what this makes possible. For example: no access to target clock? No problem, you can now use this to capture SWD-synchronous traces.

# Glitching Too!

You can, of course, also have `scope.bitbanger` trigger a voltage or clock glitch (or an EM glitch with ChipSHOUTER).

We won't demonstrate this here; the mechanics are the same as the `TIO4`-triggered glitch that we teach in our fault101 notebooks.